# Notebook 1 — Data Preprocessing & Feature Engineering
**SmartRetail: Predictive Demand Forecasting & Dynamic Pricing Engine**

### What this notebook does
1. Installs PySpark on the Colab runtime
2. Mounts Google Drive and loads the raw Retailrocket clickstream CSVs
3. Converts Unix millisecond epochs into human-readable calendar frames
4. Unions the two item-property parts and extracts category metadata
5. Engineers a deterministic float base price per SKU (prices are anonymised in this dataset)
6. Aggregates clickstream events into daily transaction counts per item
7. Builds 7-day and 30-day rolling sales-velocity features using Spark Window functions
8. Applies a strict **chronological 80/20 split** — the future 20% slice is never seen during training
9. Writes two Parquet datasets (`train/` and `test/`) and a small `raw_sample.csv` for local sanity tests

---
### Before you run
Upload the three raw files to your Google Drive under one folder and set `RAW_DATA_PATH` in the config cell below.
```
MyDrive/
└── SmartRetail/
    └── data/
        └── raw/
            ├── events.csv
            ├── item_properties_part1.csv
            └── item_properties_part2.csv
```

## Step 1 — Install PySpark

In [ ]:
!pip install pyspark==3.5.1 -q
print("PySpark installed.")

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## Step 3 — Configuration
**Only edit this cell.** All paths below derive from these two roots.

In [ ]:
# ── Edit these two lines to match your Drive layout ────────────────────────────
RAW_DATA_PATH  = "/content/drive/MyDrive/SmartRetail/data/raw/"
OUTPUT_PATH    = "/content/drive/MyDrive/SmartRetail/data/processed/"
# ───────────────────────────────────────────────────────────────────────────────

EVENTS_CSV      = RAW_DATA_PATH + "events.csv"
PROPS_CSV_1     = RAW_DATA_PATH + "item_properties_part1.csv"
PROPS_CSV_2     = RAW_DATA_PATH + "item_properties_part2.csv"

TRAIN_PARQUET   = OUTPUT_PATH + "train/"
TEST_PARQUET    = OUTPUT_PATH + "test/"
RAW_SAMPLE_CSV  = OUTPUT_PATH + "raw_sample.csv"

TRAIN_RATIO     = 0.80   # 80% past for training, 20% future for streaming simulation

print("Config ready.")
print(f"  Events      : {EVENTS_CSV}")
print(f"  Props part1 : {PROPS_CSV_1}")
print(f"  Props part2 : {PROPS_CSV_2}")
print(f"  Train out   : {TRAIN_PARQUET}")
print(f"  Test out    : {TEST_PARQUET}")

## Step 4 — Initialise SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SmartRetail-ETL")
    # Colab gives ~12 GB RAM on the standard runtime; keep driver reasonable
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "100")
    # Silence verbose INFO logs so output stays readable
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

## Step 5 — Load Raw CSVs
We infer schema on load and immediately rename columns to consistent snake_case.

In [ ]:
from pyspark.sql import functions as F

# ── Events ─────────────────────────────────────────────────────────────────────
# Schema: timestamp (unix ms), visitorid, event, itemid, transactionid
events_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(EVENTS_CSV)
)

# ── Item Properties (two parts — union immediately) ────────────────────────────
# Schema: timestamp (unix ms), itemid, property, value
props_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PROPS_CSV_1)
    .union(
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(PROPS_CSV_2)
    )
)

print("Raw row counts:")
print(f"  events          : {events_raw.count():,}")
print(f"  item_properties : {props_raw.count():,}")

events_raw.printSchema()
props_raw.printSchema()

## Step 6 — Sanitise Timestamps
The dataset stores Unix **milliseconds**. We convert to:
- `event_ts` — full timestamp (for ordering)
- `event_date` — calendar date (for daily aggregation)
- Calendar decomposition features the ML model will use: `day_of_week`, `day_of_month`, `month`, `week_of_year`

In [ ]:
events = (
    events_raw
    # Unix ms → seconds → timestamp
    .withColumn("event_ts",       F.to_timestamp(F.col("timestamp") / 1000))
    .withColumn("event_date",     F.to_date(F.col("event_ts")))
    # Calendar features for the downstream ML model
    .withColumn("day_of_week",    F.dayofweek(F.col("event_ts")).cast("int"))   # 1=Sun … 7=Sat
    .withColumn("day_of_month",   F.dayofmonth(F.col("event_ts")).cast("int"))
    .withColumn("month",          F.month(F.col("event_ts")).cast("int"))
    .withColumn("week_of_year",   F.weekofyear(F.col("event_ts")).cast("int"))
    # Drop the raw epoch column — we have everything we need from it now
    .drop("timestamp")
)

print("Date range in dataset:")
events.select(F.min("event_date").alias("earliest"), F.max("event_date").alias("latest")).show()

print("Event type breakdown:")
events.groupBy("event").count().orderBy(F.col("count").desc()).show()

## Step 7 — Extract Item Metadata from Properties

The Retailrocket dataset anonymises all property names with numeric codes. The only 
human-readable property we can rely on is `categoryid`. We extract the **most recent** 
category assignment per item (some items change category over time).

In [ ]:
from pyspark.sql.window import Window

# Keep only the categoryid rows
category_raw = props_raw.filter(F.col("property") == "categoryid")

# Some items have multiple category assignments over time.
# Pick the most recent one using a row_number window.
w_latest = Window.partitionBy("itemid").orderBy(F.col("timestamp").desc())

item_category = (
    category_raw
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .select(
        F.col("itemid"),
        F.col("value").cast("int").alias("categoryid")
    )
)

print(f"Unique items with a category: {item_category.count():,}")
item_category.show(5)

## Step 8 — Engineer Deterministic Base Prices

Retailrocket anonymises all numeric property values (encoded as `n277.200`, `n552.000`, etc.).
We cannot reconstruct real prices, so we derive a **deterministic float base price per SKU** 
using Spark's built-in `hash()` function. This produces a stable price in the realistic 
e-commerce range \[5.00, 500.00\] that is:
- Reproducible — same item always gets the same price
- Unique enough per item — the 32-bit hash space prevents clusters
- Consistent with the downstream dynamic pricing logic that will adjust it

In [ ]:
# hash() returns a signed 32-bit int; abs() + modulo maps it to [0, 49500]
# then /100 + 5.0 maps to [5.00, 500.00]
item_price = (
    events.select("itemid").distinct()
    .withColumn(
        "base_price",
        (F.abs(F.hash(F.col("itemid").cast("string"))) % 49500 / 100.0 + 5.0)
        .cast("double")
    )
)

print(f"Items with base prices: {item_price.count():,}")
print("\nBase price distribution:")
item_price.select(
    F.round(F.min("base_price"), 2).alias("min"),
    F.round(F.avg("base_price"), 2).alias("avg"),
    F.round(F.max("base_price"), 2).alias("max")
).show()

## Step 9 — Build Item Master & Join onto Events

Join category and price lookups onto the events table. Items missing a category get 
category `0` (an "unknown" bucket the `StringIndexer` in notebook 2 will handle).

In [ ]:
# Build item master: one row per item with category + base_price
item_master = (
    item_price
    .join(item_category, on="itemid", how="left")
    .withColumn("categoryid", F.coalesce(F.col("categoryid"), F.lit(0)))
)

# Enrich events with item metadata
events_enriched = events.join(item_master, on="itemid", how="left")

print("Enriched events schema:")
events_enriched.printSchema()

print(f"\nRows after join: {events_enriched.count():,}")
print(f"Null category rows: {events_enriched.filter(F.col('categoryid') == 0).count():,}")

## Step 10 — Daily Transaction Aggregation

The target variable for the forecasting model is **daily transaction count per item**.

We also carry forward the supporting event counts (views, add-to-carts) as features — 
they encode consumer-intent signals that precede purchases.

In [ ]:
# Pivot event types into separate count columns per (item, date)
daily_agg = (
    events_enriched
    .groupBy(
        "itemid",
        "event_date",
        "categoryid",
        "base_price",
        "day_of_week",
        "day_of_month",
        "month",
        "week_of_year"
    )
    .agg(
        # Target variable
        F.sum(F.when(F.col("event") == "transaction",  1).otherwise(0)).alias("daily_transactions"),
        # Supporting intent features
        F.sum(F.when(F.col("event") == "view",         1).otherwise(0)).alias("daily_views"),
        F.sum(F.when(F.col("event") == "addtocart",    1).otherwise(0)).alias("daily_addtocarts"),
        # Unique visitor count (proxy for item popularity on that day)
        F.countDistinct("visitorid").alias("daily_unique_visitors")
    )
    # View-to-cart and cart-to-purchase conversion ratios
    .withColumn(
        "view_to_cart_ratio",
        F.when(F.col("daily_views") > 0,
               F.col("daily_addtocarts") / F.col("daily_views")
        ).otherwise(0.0)
    )
    .withColumn(
        "cart_to_purchase_ratio",
        F.when(F.col("daily_addtocarts") > 0,
               F.col("daily_transactions") / F.col("daily_addtocarts")
        ).otherwise(0.0)
    )
)

print(f"Daily aggregation rows: {daily_agg.count():,}")
print("\nSample of aggregated data:")
daily_agg.orderBy("event_date").show(10, truncate=False)

## Step 11 — Rolling Sales Velocity Features

We use Spark **range-based Window functions** to compute rolling averages strictly over 
past observations (no look-ahead). The window is ordered by epoch-seconds so the 
`rangeBetween` clause works in second-level units:
- **7-day window** : `-6 × 86400` to `0` seconds relative to current row
- **30-day window** : `-29 × 86400` to `0` seconds relative to current row

Partitioned per item so each SKU's velocity is independent.

In [ ]:
SECONDS_PER_DAY = 86400

# Add epoch-seconds column for range-based window ordering
daily_agg = daily_agg.withColumn(
    "date_epoch_sec",
    F.unix_timestamp(F.col("event_date").cast("timestamp"))
)

# Window specs — partitioned per SKU, ordered by epoch seconds
w7  = (
    Window.partitionBy("itemid")
    .orderBy("date_epoch_sec")
    .rangeBetween(-6 * SECONDS_PER_DAY, 0)   # current day + 6 preceding = 7 days
)

w30 = (
    Window.partitionBy("itemid")
    .orderBy("date_epoch_sec")
    .rangeBetween(-29 * SECONDS_PER_DAY, 0)  # current day + 29 preceding = 30 days
)

daily_featured = (
    daily_agg
    .withColumn("velocity_7d",          F.avg("daily_transactions").over(w7))
    .withColumn("velocity_30d",         F.avg("daily_transactions").over(w30))
    # Also roll up view volume as a leading-indicator feature
    .withColumn("view_velocity_7d",     F.avg("daily_views").over(w7))
    # Round rolling averages for cleanliness
    .withColumn("velocity_7d",          F.round(F.col("velocity_7d"),  4))
    .withColumn("velocity_30d",         F.round(F.col("velocity_30d"), 4))
    .withColumn("view_velocity_7d",     F.round(F.col("view_velocity_7d"), 4))
    # Drop the helper epoch column — no longer needed after windows are computed
    .drop("date_epoch_sec")
)

print("Feature set after velocity engineering:")
daily_featured.printSchema()

print("\nSample rows with velocity features:")
daily_featured.orderBy("itemid", "event_date").show(15, truncate=False)

## Step 12 — Quality Checks

Before splitting, run a quick sanity pass:
- Confirm no null values in mandatory columns
- Confirm the target variable (`daily_transactions`) has a plausible distribution
- Print the full chronological date range

In [ ]:
mandatory_cols = [
    "itemid", "event_date", "categoryid", "base_price",
    "daily_transactions", "velocity_7d", "velocity_30d"
]

print("── Null check ─────────────────────────────────")
null_counts = daily_featured.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in mandatory_cols
])
null_counts.show()

print("── Target variable distribution ───────────────")
daily_featured.select(
    F.min("daily_transactions").alias("min"),
    F.round(F.avg("daily_transactions"), 4).alias("mean"),
    F.round(F.stddev("daily_transactions"), 4).alias("std"),
    F.max("daily_transactions").alias("max"),
    F.count("*").alias("total_rows")
).show()

print("── Date coverage ──────────────────────────────")
daily_featured.select(
    F.min("event_date").alias("first_date"),
    F.max("event_date").alias("last_date"),
    F.countDistinct("event_date").alias("distinct_days")
).show()

print("── Unique items ───────────────────────────────")
print(f"  SKUs in dataset: {daily_featured.select('itemid').distinct().count():,}")

## Step 13 — Chronological 80/20 Split

**Critical rule**: we split on **calendar dates**, not on random rows. 
The boundary is the date that falls at the 80th percentile of the sorted date list.

- `train/` — everything before the split date → model training in notebook 2 & 3
- `test/`  — everything from the split date onwards → streamed through Kafka in Phase 2

No row from the test window is ever passed to a model during training.

In [ ]:
# Collect sorted unique dates (small — at most ~180 dates for this dataset)
all_dates = sorted([
    row.event_date
    for row in daily_featured.select("event_date").distinct().collect()
])

n_total  = len(all_dates)
split_idx = int(n_total * TRAIN_RATIO)
split_date = all_dates[split_idx]

print(f"Total distinct dates : {n_total}")
print(f"Split index          : {split_idx}  ({TRAIN_RATIO*100:.0f}% of dates)")
print(f"Split date boundary  : {split_date}")
print(f"Training window      : {all_dates[0]}  →  {all_dates[split_idx - 1]}")
print(f"Streaming window     : {split_date}  →  {all_dates[-1]}")

train_df = daily_featured.filter(F.col("event_date") < F.lit(split_date))
test_df  = daily_featured.filter(F.col("event_date") >= F.lit(split_date))

train_count = train_df.count()
test_count  = test_df.count()

print(f"\nTrain rows : {train_count:,}  ({train_count / (train_count + test_count) * 100:.1f}%)")
print(f"Test rows  : {test_count:,}  ({test_count  / (train_count + test_count) * 100:.1f}%)")

## Step 14 — Write Outputs

- **Parquet (train & test)** — columnar, compressed, fast to read back in notebook 2 & 3
- **raw_sample.csv** — 2000 rows from the test slice; used in Phase 2 local scripts for 
  quick sanity testing without needing the full dataset on a local machine

In [ ]:
import os

# ── Parquet ────────────────────────────────────────────────────────────────────
print("Writing train Parquet...")
(
    train_df
    .orderBy("event_date", "itemid")   # keep chronological order on disk
    .write
    .mode("overwrite")
    .parquet(TRAIN_PARQUET)
)
print(f"  ✓ Train written to : {TRAIN_PARQUET}")

print("Writing test Parquet...")
(
    test_df
    .orderBy("event_date", "itemid")
    .write
    .mode("overwrite")
    .parquet(TEST_PARQUET)
)
print(f"  ✓ Test written to  : {TEST_PARQUET}")

# ── raw_sample.csv ─────────────────────────────────────────────────────────────
# Take the first 2000 rows of the test slice (chronological order)
# and write as a plain CSV that the Phase 2 local producer script can read
print("Writing raw_sample.csv...")
sample_pd = (
    test_df
    .orderBy("event_date", "itemid")
    .limit(2000)
    .toPandas()
)
sample_pd.to_csv(RAW_SAMPLE_CSV, index=False)
print(f"  ✓ Sample written to: {RAW_SAMPLE_CSV}  ({len(sample_pd)} rows)")

## Step 15 — Final Summary

In [ ]:
print("═" * 55)
print("  NOTEBOOK 1 — ETL COMPLETE")
print("═" * 55)
print()
print("Feature columns written to Parquet:")
for col_name in train_df.columns:
    print(f"  · {col_name}")
print()
print("Output locations:")
print(f"  Train Parquet  → {TRAIN_PARQUET}")
print(f"  Test Parquet   → {TEST_PARQUET}")
print(f"  raw_sample.csv → {RAW_SAMPLE_CSV}")
print()
print("What notebook 2 expects:")
print("  Read TRAIN_PARQUET for GBTRegressor training.")
print("  Target column : daily_transactions")
print("  Categorical   : itemid  (StringIndexer → itemid_index)")
print("  Categorical   : categoryid  (StringIndexer → categoryid_index)")
print()
print("What notebook 3 expects:")
print("  Read both TRAIN_PARQUET and TEST_PARQUET.")
print("  LSTM will re-aggregate into 1-min tumbling windows.")
print()
print("What Phase 2 local scripts expect:")
print("  producer.py reads raw_sample.csv row-by-row → Kafka topic.")

spark.stop()
print()
print("SparkSession closed.")